In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Predicting Titanic Survivor with a Logistic Regression Model

## 1. Preprocessing 

In [ ]:
train_set = pd.read_csv("/kaggle/input/titanic/train.csv")
test_set = pd.read_csv("/kaggle/input/titanic/test.csv")
train_set = train_set.fillna(train_set.mean(numeric_only=True))
test_set = test_set.fillna(test_set.mean(numeric_only=True))


cols_train = [i for i in range(2, 12) if i != 3 and i != 8 and i != 10]
cols_test = [i for i in range(1, 11) if i != 2 and i != 9 and i != 7]

X_train = train_set.iloc[:, cols_train]
y_train = train_set.iloc[:,1].values
X_test = test_set.iloc[:, cols_test]

print(X_train)
print(X_test)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [1,6])], remainder='passthrough')
X_train = np.array(ct.fit_transform(X_train))
X_test = np.array(ct.transform(X_test))

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)
print(X_train[:3])

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()

col_standard = [6, 9]
X_train[:,col_standard] = sc.fit_transform(X_train[:, col_standard])
X_test[:, col_standard] = sc.transform(X_test[:, col_standard])

## 2. Classification

In [ ]:
# param logistik: (max_iter=1000, random_state=42, C=0.1,penalty='l2', solver='liblinear')
from sklearn.linear_model import LogisticRegression
cl = LogisticRegression(max_iter=1000, random_state=42, C=0.1, penalty='l2', solver='liblinear')
cl.fit(X_train, y_train)
y_pred = cl.predict(X_test)

## 3. Evaluation

In [ ]:
pass_id = test_set.iloc[:,0].values

result = np.concatenate((pass_id.reshape(-1,1), y_pred.reshape(-1,1)), axis=1)

df = pd.DataFrame(result, columns=["PassengerId", "Survived"])
df.to_csv("logistic2pandas.csv", index=False)

